# Prepare real video clips on Google Drive

This notebook downloads **real** stock videos (theme: **city and people**) and
prepares them into a training dataset on your Google Drive at
`MyDrive/vidgin-data/tiny-clips/`: each video becomes a 17-frame, 64×64 clip
plus a caption in `manifest.jsonl`.

**Checkpoints vs. data, in one line:** checkpoints are the *trained model*
(brain, lives in `vidgin-checkpoints/`), while this notebook builds the
*training data* (textbook, lives in `vidgin-data/`) — training reads the
data and writes the checkpoints.

**Plan:**
1. Mount Drive, get the code
2. Configure the collection (queries, how many per query)
3. Download + prepare clips (needs a free Pexels API key, entered privately)
4. Verify the dataset and preview a clip
5. Training commands for the real-data run (do this after the Moving MNIST run finishes)


In [ ]:
# --- Mount Google Drive (your dataset lives there) ---
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")


In [ ]:
# --- Get the code (clones your repo; safe to re-run) ---
import os, subprocess
repo = "/content/Vidgin"
if not os.path.isdir(repo):
    subprocess.run(["git", "clone", "https://github.com/shahrozkalash/Vidgin.git", repo], check=True)
else:
    subprocess.run(["git", "-C", repo, "pull", "--ff-only"], check=False)
os.chdir(repo)
print("Working in:", os.getcwd())

# cv2 ships with Colab; imageio is the fallback decoder for the prep script.
subprocess.run(["pip", "install", "-q", "numpy", "pytest", "pillow", "imageio", "imageio-ffmpeg"], check=True)
print("Setup done.")


## Configuration — "City and people" theme

Six Pexels searches, 500 videos each → roughly **3,000 clips** after the
quality filters (each kept clip is ~200 KB, so ~600 MB on your Drive).
Edit the list freely — any comma-free search phrases work.


In [ ]:
# --- What to collect ---
QUERIES = [
    "people walking city street",
    "crowd crossing street",
    "city traffic aerial",
    "subway station people",
    "busy market street",
    "city timelapse traffic",
]
PER_QUERY = 500
OUT = "/content/drive/MyDrive/vidgin-data/tiny-clips"
print(f"{len(QUERIES)} queries x {PER_QUERY} = ~{len(QUERIES) * PER_QUERY} videos to fetch")
print("Output:", OUT)


## Download and prepare the clips

You need a **free** Pexels API key (no card): sign up at
[pexels.com/api](https://www.pexels.com/api/), copy the key, and paste it in
the prompt below. `getpass` keeps it out of the notebook text and out of chat —
it lives only in this session's memory.
This cell takes a while (downloading ~3,000 videos); it prints progress as it goes.


In [ ]:
# --- Run the collector ---
import getpass, os
key = getpass.getpass("Pexels API key: ").strip()
assert key, "No key entered — get a free one at https://www.pexels.com/api/"
os.environ["PEXELS_API_KEY"] = key

!python scripts/prepare_clips.py --source pexels \
    --queries "{','.join(QUERIES)}" \
    --per-query {PER_QUERY} \
    --out "{OUT}" --frames 17 --size 64

# No-key alternative (uncomment to use direct video URLs instead of Pexels):
# !python scripts/prepare_clips.py --source urls --urls-file urls.txt --out "{OUT}" --frames 17 --size 64


## Verify the dataset

Load what was prepared and look at the very first clip.


In [ ]:
# --- Check the dataset + preview one clip ---
import numpy as np
import matplotlib.pyplot as plt
from vidgen.data.clips import FolderClipDataset

ds = FolderClipDataset(OUT)
print("clips:", len(ds))
t, caption = ds[0]
print("sample:", tuple(t.shape), t.dtype, f"range=[{t.min():.2f},{t.max():.2f}]")
print("caption:", caption)
assert tuple(t.shape) == (3, 17, 64, 64), "unexpected clip geometry"

frames = ((t.permute(1, 2, 3, 0).numpy() * 0.5 + 0.5) * 255).clip(0, 255).astype(np.uint8)
fig, axes = plt.subplots(1, 17, figsize=(17, 1.4))
for ax, fr in zip(axes, frames):
    ax.imshow(fr); ax.axis("off")
plt.tight_layout(); plt.show()


## Train on these clips (after the Moving MNIST run finishes)

Real-clip training is a **fresh run**: the VAE trains from scratch on your
city/people clips, then the DiT trains in that VAE's latent space.
Checkpoints go to their own Drive folders so they never mix with the
Moving MNIST ones.

**VAE (~1–2 h on a T4):**
```bash
!python -m vidgen.train_vae --device cuda --steps 4000 --batch-size 16 \
    --data-dir /content/drive/MyDrive/vidgin-data/tiny-clips \
    --ckpt-dir /content/drive/MyDrive/vidgin-checkpoints/real-vae --ckpt-every 500 \
    --log /content/drive/MyDrive/vidgin-checkpoints/real-vae/train.jsonl
```

**DiT (~4–8 h on a T4, run after the VAE finishes):**
```bash
!python -m vidgen.train --device cuda --steps 15000 --batch-size 16 \
    --data-dir /content/drive/MyDrive/vidgin-data/tiny-clips \
    --vae-ckpt /content/drive/MyDrive/vidgin-checkpoints/real-vae/latest.pt \
    --ckpt-dir /content/drive/MyDrive/vidgin-checkpoints/real-dit --ckpt-every 500 \
    --log /content/drive/MyDrive/vidgin-checkpoints/real-dit/train.jsonl
```

Then generate a clip the usual way (`sample.py`) — prompts like
*"people crossing a city street"* now come from captions like these,
not from Moving MNIST digits.
